# Project 1 for CSCI 331
### Mehtab Mahir


# Case #001: The Mystery of the Missing Inventory
### Scenario
In the AdventureWorks warehouse in Seattle, a high-value inventory item has gone missing on **October 12, 2023**.
Our job is to investigate the disappearance by:
- Identifying the missing inventory
- Tracing transaction history
- Identifying the employee involved
- Confirming involvement via communication logs

## Step 1: Search for Inventory Activity on the Incident Date

In [2]:
SELECT * 
FROM Production.ProductInventory 
WHERE ModifiedDate = '2023-10-12';

(0 rows affected)

Total execution time: 00:00:00.005

ProductID,LocationID,Shelf,Bin,Quantity,rowguid,ModifiedDate


Since no rows were found, we simulated the missing item by inserting a custom entry for **ProductID = 776**.

In [3]:
INSERT INTO Production.ProductInventory (ProductID, LocationID, Shelf, Bin, Quantity, ModifiedDate)
VALUES (776, 3, 'A', 1, 0, '2023-10-12');

(1 row affected)

Total execution time: 00:00:00.039

## Step 2: Trace the Inventory Item in the Transaction History

In [4]:
SELECT *
FROM Production.TransactionHistory
WHERE ProductID = 776 AND TransactionDate = '2023-10-12';

(0 rows affected)

Total execution time: 00:00:00.008

TransactionID,ProductID,ReferenceOrderID,ReferenceOrderLineID,TransactionDate,TransactionType,Quantity,ActualCost,ModifiedDate


Since no transaction was found, we inserted one to simulate suspicious movement of the item.

In [5]:
INSERT INTO Production.TransactionHistory (
    ProductID, ReferenceOrderID, ReferenceOrderLineID, TransactionDate,
    TransactionType, Quantity, ActualCost, ModifiedDate)
VALUES (776, 99999, 1, '2023-10-12', 'W', -1, 1000.00, '2023-10-12');

(1 row affected)

Total execution time: 00:00:00.039

## Step 3: Identify Potential Employees

In [6]:
SELECT TOP 5 e.BusinessEntityID, p.FirstName, p.LastName
FROM HumanResources.Employee AS e
JOIN Person.Person AS p ON e.BusinessEntityID = p.BusinessEntityID;

(5 rows affected)

Total execution time: 00:00:00.025

BusinessEntityID,FirstName,LastName
263,Jean,Trenary
78,Reuben,D'sa
242,Deborah,Poe
125,Matthias,Berndt
278,Garrett,Vargas


We selected **George Li (BusinessEntityID = 170)** as the suspect based on his role and access.

## Step 4: Confirm Involvement via Communication Logs

In [7]:
INSERT INTO dbo.CommunicationLogs (EmployeeID, Message, Date)
VALUES (170, 'Inventory transfer must be untraceable before audit.', '2023-10-12');

(1 row affected)

Total execution time: 00:00:00.040

## Final Query: Match Employee to Communication
This confirms George Li's suspicious message on the day of the missing inventory.

In [8]:
SELECT e.BusinessEntityID, p.FirstName, p.LastName, cl.Message
FROM HumanResources.Employee e
JOIN Person.Person p ON e.BusinessEntityID = p.BusinessEntityID
JOIN dbo.CommunicationLogs cl ON e.BusinessEntityID = cl.EmployeeID
WHERE cl.Date = '2023-10-12';

(1 row affected)

Total execution time: 00:00:00.024

BusinessEntityID,FirstName,LastName,Message
170,George,Li,Inventory transfer must be untraceable before audit.


# Case #002: The Data Breach Dilemma
### Scenario
In the London-based WideWorldImporters data center, a data breach was detected on **November 5, 2023**. Sensitive customer information may have been accessed.

Our mission:
- Identify suspicious security activity
- Check for sensitive data access
- Identify the responsible employee
- Confirm involvement via email communications

## Step 1: Log the Security Event

In [2]:
CREATE TABLE dbo.SecurityLogs (
    LogID INT IDENTITY(1,1) PRIMARY KEY,
    EmployeeID INT,
    Action NVARCHAR(MAX),
    Date DATETIME
);

Commands completed successfully.

Total execution time: 00:00:00.009

In [3]:
INSERT INTO dbo.SecurityLogs (EmployeeID, Action, Date)
VALUES (4, 'Downloaded encrypted customer data outside of work hours', '2023-11-05');

(1 row affected)

Total execution time: 00:00:00.007

## Step 2: Log Access to Customer Data

In [4]:
CREATE TABLE dbo.AccessRecords (
    AccessID INT IDENTITY(1,1) PRIMARY KEY,
    PersonID INT,
    TableAccessed NVARCHAR(100),
    AccessType NVARCHAR(50),
    DateAccessed DATETIME
);

Commands completed successfully.

Total execution time: 00:00:00.007

In [5]:
INSERT INTO dbo.AccessRecords (PersonID, TableAccessed, AccessType, DateAccessed)
VALUES (4, 'Sales.Customers', 'SELECT', '2023-11-05');

(1 row affected)

Total execution time: 00:00:00.008

## Step 3: Identify the Suspect

In [6]:
SELECT PersonID, FullName, PreferredName
FROM Application.People
WHERE PersonID = 4;

(1 row affected)

Total execution time: 00:00:00.015

PersonID,FullName,PreferredName
4,Isabella Rupp,Isabella


**Suspect Identified:** Isabella Rupp (PersonID: 4)

## Step 4: Confirm via Email Evidence

In [7]:
CREATE TABLE dbo.EmailCommunications (
    EmailID INT IDENTITY(1,1) PRIMARY KEY,
    PersonID INT,
    Subject NVARCHAR(255),
    Body NVARCHAR(MAX),
    DateSent DATETIME
);

Commands completed successfully.

Total execution time: 00:00:00.012

In [8]:
INSERT INTO dbo.EmailCommunications (PersonID, Subject, Body, DateSent)
VALUES (4, 'Re: Exported Contact List',
        'I’ve extracted the full customer list — let me know where to upload it.',
        '2023-11-05');

(1 row affected)

Total execution time: 00:00:00.009

## Final Query: Link Person to Email

In [9]:
SELECT p.FullName, ec.Subject, ec.Body, ec.DateSent
FROM Application.People AS p
JOIN dbo.EmailCommunications AS ec ON p.PersonID = ec.PersonID
WHERE ec.DateSent = '2023-11-05';

(1 row affected)

Total execution time: 00:00:00.022

FullName,Subject,Body,DateSent
Isabella Rupp,Re: Exported Contact List,I’ve extracted the full customer list — let me know where to upload it.,2023-11-05 00:00:00.000


# Case #003: The Vanishing Vehicles
### Scenario
Several high-value vehicles have gone missing from the EuropeanCarsDistributor on **August 10, 2014**. Your job is to investigate the disappearance and identify the responsible party.
We are simulating this case using the **AdventureWorks2017** database.

## Step 1: Identify Inventory Activity

In [1]:
SELECT ProductID, LocationID, Shelf, Bin, Quantity
FROM Production.ProductInventory
WHERE ModifiedDate = '2014-08-10';

(3 rows affected)

Total execution time: 00:00:00.010

ProductID,LocationID,Shelf,Bin,Quantity
343,1,E,6,568
343,50,S,3,606
343,60,S,2,499


## Step 2: Check Transaction History

In [2]:
SELECT *
FROM Production.TransactionHistory
WHERE ProductID = 343 AND TransactionDate = '2014-08-10';

(0 rows affected)

Total execution time: 00:00:00.010

TransactionID,ProductID,ReferenceOrderID,ReferenceOrderLineID,TransactionDate,TransactionType,Quantity,ActualCost,ModifiedDate


In [3]:
INSERT INTO Production.TransactionHistory (
    ProductID, ReferenceOrderID, ReferenceOrderLineID, TransactionDate,
    TransactionType, Quantity, ActualCost, ModifiedDate
)
VALUES (
    343, 88888, 1, '2014-08-10',
    'W', -1, 75000.00, '2014-08-10'
);

(1 row affected)

Total execution time: 00:00:00.012

## Step 3: Identify the Suspect

In [4]:
SELECT TOP 5 e.BusinessEntityID, p.FirstName, p.LastName
FROM HumanResources.Employee AS e
JOIN Person.Person AS p ON e.BusinessEntityID = p.BusinessEntityID;

(5 rows affected)

Total execution time: 00:00:00.010

BusinessEntityID,FirstName,LastName
263,Jean,Trenary
78,Reuben,D'sa
242,Deborah,Poe
125,Matthias,Berndt
278,Garrett,Vargas


In [5]:
INSERT INTO dbo.CommunicationLogs (EmployeeID, Message, Date)
VALUES (278, 'Vehicle shipment must be off the books before inventory closes.', '2014-08-10');

(1 row affected)

Total execution time: 00:00:00.009

## Step 4: Confirm Involvement via Communication Logs

In [6]:
SELECT e.BusinessEntityID, p.FirstName, p.LastName, cl.Message
FROM HumanResources.Employee e
JOIN Person.Person p ON e.BusinessEntityID = p.BusinessEntityID
JOIN dbo.CommunicationLogs cl ON e.BusinessEntityID = cl.EmployeeID
WHERE cl.Date = '2014-08-10';

(1 row affected)

Total execution time: 00:00:00.010

BusinessEntityID,FirstName,LastName,Message
278,Garrett,Vargas,Vehicle shipment must be off the books before inventory closes.


# Case #004: The Financial Fraud Firewall
### Scenario
Suspicious price changes have been detected in the financial systems of AdventureWorks on **January 17, 2014**.
You're tasked with finding the modified prices and identifying the employee behind the fraudulent activity.

## Step 1: Check for Price Changes on the Incident Date

In [1]:
SELECT * 
FROM Production.ProductListPriceHistory
WHERE ModifiedDate = '2014-01-17';

(0 rows affected)

Total execution time: 00:00:00.008

In [2]:
INSERT INTO Production.ProductListPriceHistory (
    ProductID, StartDate, EndDate, ListPrice, ModifiedDate
)
VALUES (
    800, '2014-01-17', NULL, 1999.99, '2014-01-17'
);

(1 row affected)

Total execution time: 00:00:00.014

## Step 2: Identify Potential Employees

In [3]:
SELECT TOP 5 e.BusinessEntityID, p.FirstName, p.LastName
FROM HumanResources.Employee AS e
JOIN Person.Person AS p ON e.BusinessEntityID = p.BusinessEntityID;

(5 rows affected)

Total execution time: 00:00:00.010

BusinessEntityID,FirstName,LastName
263,Jean,Trenary
78,Reuben,D'sa
242,Deborah,Poe
125,Matthias,Berndt
278,Garrett,Vargas


**Selected Suspect:** Deborah Poe (BusinessEntityID: 242)

## Step 3: Confirm with Communication Logs

In [4]:
INSERT INTO dbo.CommunicationLogs (EmployeeID, Message, Date)
VALUES (242, 'Price hike needs to be undocumented. Accounting can’t know.', '2014-01-17');

(1 row affected)

Total execution time: 00:00:00.009

## Step 4: Confirm Involvement via Message Lookup

In [5]:
SELECT e.BusinessEntityID, p.FirstName, p.LastName, cl.Message
FROM HumanResources.Employee e
JOIN Person.Person p ON e.BusinessEntityID = p.BusinessEntityID
JOIN dbo.CommunicationLogs cl ON e.BusinessEntityID = cl.EmployeeID
WHERE cl.Date = '2014-01-17';

(1 row affected)

Total execution time: 00:00:00.010

BusinessEntityID,FirstName,LastName,Message
242,Deborah,Poe,Price hike needs to be undocumented. Accounting can’t know.


# Case #005: The Silent Salesperson
### Scenario
Stephen Jiang, a salesperson, has not logged any sales for over four months.
You’ve been asked to investigate whether he is ghosting the system or hiding his inactivity.

## Step 1: Find Salespeople with No Activity in the Past 90 Days

In [1]:
SELECT sp.BusinessEntityID, p.FirstName, p.LastName
FROM Sales.SalesPerson sp
JOIN HumanResources.Employee e ON sp.BusinessEntityID = e.BusinessEntityID
JOIN Person.Person p ON e.BusinessEntityID = p.BusinessEntityID
LEFT JOIN Sales.SalesOrderHeader soh
    ON sp.BusinessEntityID = soh.SalesPersonID
    AND soh.OrderDate >= DATEADD(DAY, -90, GETDATE())
WHERE soh.SalesOrderID IS NULL;

(5 rows affected)

Total execution time: 00:00:00.010

BusinessEntityID,FirstName,LastName
274,Stephen,Jiang
275,Michael,Blythe
276,Linda,Mitchell
277,Jillian,Carson
278,Garrett,Vargas


## Step 2: Use OUTER APPLY to Find Most Recent Sale

In [2]:
SELECT p.FirstName, p.LastName, recent.OrderDate, recent.TotalDue
FROM Person.Person p
JOIN Sales.SalesPerson sp ON p.BusinessEntityID = sp.BusinessEntityID
OUTER APPLY (
    SELECT TOP 1 soh.OrderDate, soh.TotalDue
    FROM Sales.SalesOrderHeader soh
    WHERE soh.SalesPersonID = sp.BusinessEntityID
    ORDER BY soh.OrderDate DESC
) recent
WHERE p.BusinessEntityID = 274;

(1 row affected)

Total execution time: 00:00:00.010

FirstName,LastName,OrderDate,TotalDue
Stephen,Jiang,2014-05-01 00:00:00.000,39933.1824


## Step 3: Insert Suspicious Communication Log

In [3]:
INSERT INTO dbo.CommunicationLogs (EmployeeID, Message, Date)
VALUES (274, 'Let’s keep things quiet about my pipeline status this quarter.', '2014-09-09');

(1 row affected)

Total execution time: 00:00:00.009

# Case #006: The Vendor Kickback Scheme
### Scenario
Superior Bicycles appears to be receiving a disproportionately high number of purchase orders.
You’ve been tasked with investigating if a purchasing employee is favoring this vendor for personal gain.

## Step 1: Find Vendors with the Highest Purchase Totals

In [1]:
SELECT poh.VendorID, v.Name AS VendorName, SUM(poh.TotalDue) AS TotalSpent
FROM Purchasing.PurchaseOrderHeader poh
JOIN Purchasing.Vendor v ON poh.VendorID = v.BusinessEntityID
GROUP BY poh.VendorID, v.Name
ORDER BY TotalSpent DESC;

(1 row affected)

Total execution time: 00:00:00.010

VendorID,VendorName,TotalSpent
1576,Superior Bicycles,5034267.0


## Step 2: Identify Employee Responsible for Most Orders to This Vendor

In [2]:
SELECT poh.EmployeeID, COUNT(*) AS OrdersPlaced, SUM(poh.TotalDue) AS TotalSpent
FROM Purchasing.PurchaseOrderHeader poh
WHERE poh.VendorID = 1576
GROUP BY poh.EmployeeID
ORDER BY TotalSpent DESC;

(5 rows affected)

Total execution time: 00:00:00.010

EmployeeID,OrdersPlaced,TotalSpent
253,5,503426.674
255,5,503426.674
256,5,503426.674
257,5,503426.674
258,5,503426.674


## Step 3: Insert Communication Log Implicating EmployeeID 255

In [3]:
INSERT INTO dbo.CommunicationLogs (EmployeeID, Message, Date)
VALUES (255, 'Superior Bicycles will make it worth our while if we keep the orders coming.', '2014-07-22');

(1 row affected)

Total execution time: 00:00:00.005

# Case #007: The Fabricated Freight Fees
### Scenario
A spike in shipping charges was flagged on August 5, 2014.
You must investigate unusually high freight fees and identify who may have submitted a fraudulent purchase order.

## Step 1: Search for Purchase Orders with High Freight Percentage

In [1]:
SELECT PurchaseOrderID, EmployeeID, VendorID, SubTotal, Freight, (Freight / SubTotal) * 100 AS FreightPercent
FROM Purchasing.PurchaseOrderHeader
WHERE Freight > 0.2 * SubTotal
AND OrderDate = '2014-08-05';

(0 rows affected)

Total execution time: 00:00:00.010

PurchaseOrderID,EmployeeID,VendorID,SubTotal,Freight,FreightPercent


## Step 2: Insert Suspicious Purchase Order with High Freight

In [2]:
INSERT INTO Purchasing.PurchaseOrderHeader (
    RevisionNumber, Status, EmployeeID, VendorID, ShipMethodID,
    OrderDate, ShipDate, SubTotal, TaxAmt, Freight, ModifiedDate
)
VALUES (
    1, 1, 278, 1492, 3,
    '2014-08-05', NULL, 4000.00, 320.00, 2500.00, '2014-08-05'
);

(1 row affected)

Total execution time: 00:00:00.022

## Step 3: Log Suspicious Communication from Garrett Vargas

In [3]:
INSERT INTO dbo.CommunicationLogs (EmployeeID, Message, Date)
VALUES (278, 'The vendor doesn’t care what the freight says — just inflate it.', '2014-08-05');

(1 row affected)

Total execution time: 00:00:00.006

## Step 4: Confirm Involvement with Communication Lookup

In [4]:
SELECT e.BusinessEntityID, p.FirstName, p.LastName, cl.Message
FROM HumanResources.Employee e
JOIN Person.Person p ON e.BusinessEntityID = p.BusinessEntityID
JOIN dbo.CommunicationLogs cl ON e.BusinessEntityID = cl.EmployeeID
WHERE cl.Date = '2014-08-05';

(1 row affected)

Total execution time: 00:00:00.008

BusinessEntityID,FirstName,LastName,Message
278,Garrett,Vargas,The vendor doesn’t care what the freight says — just inflate it.
